## Setup

In [ ]:
# pip install nltk textblob wordcloud vaderSentiment

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud, STOPWORDS

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

## Import Data

In [ ]:
df = pd.read_csv("Tweets.csv")
df = df[['tweet_id', 'airline_sentiment', 'airline', 'text']]

df.head()

In [ ]:
LABELS = ["negative", "neutral", "positive"]

## EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))

df['airline_sentiment'].value_counts().reindex(LABELS).plot.bar(
    ax=axes[0], color=['maroon', 'gray', 'green'])
axes[0].set_title("Overall Sentiment (Human Label)")

pd.crosstab(df["airline"], df['airline_sentiment'], normalize="index")[LABELS].plot.barh(
    stacked=True, ax=axes[1], color=['maroon', 'gray', 'green'])
axes[1].set_title("Sentiment Mix per Airline")
plt.show()

In [ ]:
def clean_tweet(text):
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # URLs
    text = re.sub(r"@\w+", " ", text)               # @mentions (airline handles)
    text = text.replace("&amp;", "&").replace("#", "")
    return re.sub(r"\s+", " ", text).strip()

df['clean'] = df["text"].apply(clean_tweet)
df[['text', 'clean']].sample(5)

### WordCloud

In [ ]:
tweets = df['clean'].str.cat(sep=" ")

all_stopwords = STOPWORDS

wordcloud = WordCloud(
    background_color = 'white',
    stopwords = all_stopwords,
    max_words = 200,
    max_font_size = 50,
    scale = 3,
    random_state = 1
).generate(tweets)

plt.figure(figsize=(12,12))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

## Train/Test Split

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=42,
                                stratify=df["airline_sentiment"])

y_train = train['airline_sentiment'].to_numpy()
y_test = test['airline_sentiment'].to_numpy()

print(len(train), 'train data; ', len(test), 'test data')

results, preds = {}, {}

In [ ]:
def evaluate(name, y_pred):
    y_pred=np.asarray(y_pred)
    preds['name'] = y_pred

    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, labels=LABELS, average="macro") 
    }

    print(f"==={name}===")
    print(classification_report(y_test, y_pred, labels=LABELS, digits=3, zero_division=0))

## Model 1: VADER - lexicon + rules, built for social media 

#### VADER Examples

In [ ]:
vader = SentimentIntensityAnalyzer()

examples = [
    "The flight was good.",
    "The flight was GOOD!!!",
    "The flight was not good.",
    "The flight was very good, but the food was terrible.",
    "Lost my bag again :(",
    "Thanks for the upgrade :)",
]

In [ ]:
pd.DataFrame(
    [{"text": s, **vader.polarity_scores(s)} for s in examples]
).round(3)

### Main Model

In [ ]:
def vader_label(text, threshold=0.2):
    c = vader.polarity_scores(text)['compound']

    if c >= threshold:
        return "positive"
    elif c <= -threshold:
        return "negative"
    else:
        return "neutral"

In [ ]:
evaluate("VADER", test['clean'].apply(vader_label))

In [ ]:
results

## Model 2: TextBlob - General Purpose Lexicon

#### sentiment - polarity;  subjectivity

In [ ]:
examples = [
    "The flight was good.",
    "The flight was GOOD!!!",
    "The flight was not good.",
    "The flight was very good, but the food was terrible.",
    "Lost my bag again :(",
    "Thanks for the upgrade :)",
    "So great! So life changing!"
]

### TextBlob Examples

In [ ]:
pd.DataFrame(
    [{
        "text": s,
        "polarity": TextBlob(s).sentiment.polarity,
        "subjectivity": TextBlob(s).sentiment.subjectivity}
        for s in examples]
)

### Main Model

In [ ]:
def textblob_label(text, threshold=0.2):
    p = TextBlob(text).sentiment.polarity
    if p >= threshold:
        return "positive"
    elif p <= -threshold:
        return "negative"
    else:
        return "neutral"

In [ ]:
evaluate("TextBlob", test['clean'].apply(textblob_label))